# P02: Applications and advanced tips

Loops and conditionals are how you process trial after trial. Below is a
signal-detection-based example that analyzes responses across a block of trials, and we'll introduce some looping helpers you will use/encounter a lot (`enumerate`, `zip`, comprehensions). Last, we'll look at two loop bugs that do not raise an error.

## Classifying a block of trials

In [1]:
# a few trials, each recorded as (stimulus_present, responded_yes)
trials = [(True, True), (True, False), (False, True), (False, False), (True, True)]

# tally the four signal-detection outcomes by looping over the trials
counts = {"hit": 0, "miss": 0, "false alarm": 0, "correct rejection": 0}
for present, said_yes in trials:          # tuple unpacking right in the for-line
    if present and said_yes:
        counts["hit"] += 1
    elif present and not said_yes:
        counts["miss"] += 1
    elif said_yes:
        counts["false alarm"] += 1
    else:
        counts["correct rejection"] += 1

print(counts)

{'hit': 2, 'miss': 1, 'false alarm': 1, 'correct rejection': 1}


## `enumerate` and `zip`: two very common loop helpers 

In [4]:
rts = [0.41, 0.52, 0.38, 0.66]
accuracy = [1, 1, 0, 1]

# zip walks two (or more) lists together; enumerate hands you a counter
for i, (rt, correct) in enumerate(zip(rts, accuracy)):
    print(f"trial {i}: rt={rt}s correct={bool(correct)}")

trial 0: rt=0.41s correct=True
trial 1: rt=0.52s correct=True
trial 2: rt=0.38s correct=False
trial 3: rt=0.66s correct=True


## List comprehensions: a compact loop and a filter

In [5]:
# keep only the reaction times from correct trials
# read this as "add the rt to the correct_rts list if the subjects accuracy was ok"
correct_rts = [rt for rt, ok in zip(rts, accuracy) if ok]
print(correct_rts)

[0.41, 0.52, 0.66]


## A bounded adaptive staircase

In [6]:
import random
rng = random.Random(0)

# a simple 1-up/1-down staircase that nudges contrast toward a threshold
contrast = 0.5
step = 0.05
for trial in range(20):                          # avoid an open ended 'while' loop here
    p_correct = contrast / (contrast + 0.1)      # toy psychometric function
    correct = rng.random() < p_correct
    contrast = max(0.0, contrast - step) if correct else contrast + step

print(f"final contrast estimate: {contrast:.3f}")

final contrast estimate: 0.300


:::{admonition} Advanced tips: loop bugs that do not raise errors
:class: tip
Two classics to be aware of:

* **Infinite `while` loops.** A staircase written as `while not converged:` with no
  hard cap on trials will run forever if the subject never reaches the criterion.
  Always bound adaptive procedures with a maximum trial count (as in the `for`
  loop above).
* **Off-by-one in epoch windows.** `range(start, stop)` *excludes* `stop`. If you
  want an epoch that includes both endpoints `[onset, onset + 200]`, you need
  `range(onset, onset + 201)`. Getting this wrong shortens every epoch by one
  sample (and may not throw an error, but just subtly truncate your data).
:::